In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

In [2]:
ROOT = Path.cwd().parents[1]
DATA_DIR = ROOT / "data"

df = pd.read_csv(DATA_DIR / "duolingo_flagship_v5.csv")
split_users = pd.read_csv(DATA_DIR / "split_users.csv")

In [3]:

cv_users = split_users.loc[split_users["split"] == "cv", "user_id"]
cv_df = df[df["user_id"].isin(cv_users)].copy()

features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log",
]

X = cv_df[features]
y = cv_df["p_recall"]
groups = cv_df["user_id"]

gkf = GroupKFold(n_splits=5)

print(len(cv_df), cv_df["user_id"].nunique())

14438 2125


In [4]:
depths = [2, 3, 4, 5, 6, 8, 10, None]
results = []

for depth in depths:
    fold_rmses = []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = DecisionTreeRegressor(max_depth=depth,random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, pred))
        fold_rmses.append(rmse)

    results.append({
        "max_depth": depth,
        "mean_rmse": np.mean(fold_rmses),
        "std_rmse": np.std(fold_rmses)
    })

pd.DataFrame(results)

,max_depth,mean_rmse,std_rmse
0,2.0,0.274308,0.013687
1,3.0,0.274206,0.013786
2,4.0,0.274545,0.013907
3,5.0,0.275970,0.013579
4,6.0,0.276961,0.012681
5,8.0,0.281055,0.012505
6,10.0,0.288598,0.014907
7,NaN,0.370018,0.011001


In [7]:
leaf_sizes = [1, 2, 5, 10, 20, 50, 100,150,200,300,500,750,1000,2000,4000,8000,16000]
results = []

for leaf_size in leaf_sizes:
    fold_rmses = []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = DecisionTreeRegressor(min_samples_leaf=leaf_size,random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, pred))
        fold_rmses.append(rmse)

    results.append({
        "min_samples_size": leaf_size,
        "mean_rmse": np.mean(fold_rmses),
        "std_rmse": np.std(fold_rmses)
    })

pd.DataFrame(results)

,min_samples_size,mean_rmse,std_rmse
0,1,0.370018,0.011001
1,2,0.335282,0.013886
2,5,0.307305,0.014773
3,10,0.292902,0.013476
4,20,0.283631,0.013547
5,50,0.278461,0.013038
6,100,0.276043,0.012952
7,150,0.275743,0.013478
8,200,0.275150,0.013479
9,300,0.274699,0.013541


In [8]:
split_sizes = [2, 10, 50, 100, 500, 1000, 2000, 4000, 8000]

results_split = []

for split_size in split_sizes:
    fold_rmses = []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = DecisionTreeRegressor(min_samples_split=split_size,random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, pred))
        fold_rmses.append(rmse)

    results_split.append({
        "min_samples_split": split_size,
        "mean_rmse": np.mean(fold_rmses),
        "std_rmse": np.std(fold_rmses)
    })

pd.DataFrame(results_split)

,min_samples_split,mean_rmse,std_rmse
0,2,0.370018,0.011001
1,10,0.332214,0.011522
2,50,0.299274,0.011846
3,100,0.288663,0.011014
4,500,0.277518,0.013110
5,1000,0.275761,0.013838
6,2000,0.274852,0.014082
7,4000,0.274105,0.013991
8,8000,0.274075,0.013892


In [9]:
alphas = [0.0, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2]

results_ccp = []

for alpha in alphas:
    fold_rmses = []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = DecisionTreeRegressor(ccp_alpha=alpha,random_state=42)
        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, pred))
        fold_rmses.append(rmse)

    results_ccp.append({
        "ccp_alpha": alpha,
        "mean_rmse": np.mean(fold_rmses),
        "std_rmse": np.std(fold_rmses)
    })

ccp_results_df = pd.DataFrame(results_ccp)
ccp_results_df

,ccp_alpha,mean_rmse,std_rmse
0,0.000000,0.370018,0.011001
1,0.000001,0.369978,0.010901
2,0.000010,0.362782,0.011454
3,0.000100,0.275131,0.013860
4,0.001000,0.275289,0.014628
5,0.010000,0.275289,0.014628



Unrestricted DecisionTreeRegressor strongly overfit:
- RMSE ≈ 0.3700

Complexity control substantially improved generalization:
- max_depth=3 → RMSE ≈ 0.2742
- min_samples_leaf=2000 → RMSE ≈ 0.2739
- ccp_alpha=1e-4 → RMSE ≈ 0.2751

Key idea:
- Pre-pruning: max_depth, min_samples_split, min_samples_leaf
- Post-pruning: ccp_alpha
- More regularization → bias ↑, variance ↓

Conclusion:
The tree family was not inherently poor, the unrestricted tree was too complex for this noisy target.